# 13. Autoencoders
**Use case:** Oil & Gas — Gas compressor sensor anomaly detection for predictive maintenance

Gas compression stations keep pipelines flowing, and an unplanned compressor trip can cut throughput and cost a great deal per day in lost deliveries and emergency repairs. Each unit streams vibration, temperature, pressure, flow and motor current readings, but real failures are rare and seldom well labelled. Simple fixed limits on each sensor miss faults where every reading is in range but the readings no longer agree with each other. Success means catching developing faults hours before a trip, with few false alarms, and pointing technicians to the likely sensor or component.

> 💡 A seasoned compressor technician can glance at the gauges and say 'with this load, I expect these readings'. When the gauges disagree with what they would expect, they know something is off - the autoencoder learns that same 'expected picture' from healthy history.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## EASY: Compress 4 sensors to 1 number and rebuild
Use a hand-made linear encoder and decoder in numpy to see how reconstruction error exposes a bearing fault.

| 5W + H | |
|---|---|
| **Who** | A reliability engineer new to machine learning. |
| **What** | Eight healthy compressor readings and two new readings, one with a vibration spike. |
| **When** | Before training any network, to understand the core idea. |
| **Where** | A short numpy script. |
| **Why** | Seeing that 'cannot rebuild it' equals 'unusual' makes autoencoder alarms easy to trust. |
| **How** | Encode = average the four sensors into a load number, decode = copy it back to four sensors, compare. |

**🧠 ML concepts:** Encoder and decoder, Bottleneck code, Reconstruction error as anomaly score, Threshold from healthy data, Per-sensor error attribution  
**🐍 Python concepts:** 2-D numpy arrays, Defining a function that returns a tuple, List comprehension, Dictionaries and .items(), Conditional expression, np.argmax for the index of the largest value

In [ ]:
import numpy as np

# 8 healthy readings from a gas compressor, already scaled to 0-1:
# columns = [vibration, temperature, pressure, flow]
# 🐍 PY: on a healthy machine all 4 sensors rise and fall together with the load
healthy = np.array([
    [0.20, 0.22, 0.21, 0.19],
    [0.40, 0.41, 0.39, 0.42],
    [0.55, 0.53, 0.56, 0.54],
    [0.70, 0.72, 0.69, 0.71],
    [0.30, 0.29, 0.31, 0.30],
    [0.85, 0.84, 0.86, 0.85],
    [0.60, 0.62, 0.59, 0.61],
    [0.45, 0.44, 0.46, 0.45],
])

# 🧠 ML: an autoencoder squeezes inputs to a smaller 'code' and rebuilds them
# 🧠 ML: here the encoder is one weight vector: code = average of the 4 sensors (1 number = 'load')
encoder = np.array([0.25, 0.25, 0.25, 0.25])
# 🧠 ML: the decoder spreads the code back out: healthy machine -> every sensor near the load level
decoder = np.array([1.0, 1.0, 1.0, 1.0])

def reconstruct(x):
    # 🐍 PY: x @ encoder is a dot product -> one number; times decoder broadcasts back to 4 numbers
    code = x @ encoder
    return code, code * decoder

# 🐍 PY: list comprehension computes the reconstruction error for every healthy row
errors = [np.mean((row - reconstruct(row)[1]) ** 2) for row in healthy]
# 🧠 ML: alarm threshold = a bit above the worst error seen on healthy data
threshold = max(errors) * 3
print("Healthy reconstruction errors:", np.round(errors, 5))
print(f"Alarm threshold: {threshold:.5f}\n")

# New readings: one normal, one where vibration spikes but load is normal (bearing wear)
# 🐍 PY: a dict maps a description to the reading
new = {
    "normal load": np.array([0.50, 0.51, 0.49, 0.50]),
    "bearing wear": np.array([0.90, 0.50, 0.49, 0.50]),
}
# 🐍 PY: .items() yields (key, value) pairs we can unpack in the loop header
for name, x in new.items():
    code, rebuilt = reconstruct(x)
    err = np.mean((x - rebuilt) ** 2)
    # 🐍 PY: conditional expression picks a label in one line
    status = "ANOMALY" if err > threshold else "ok"
    print(f"{name:<13} code={code:.3f} rebuilt={np.round(rebuilt, 2)} error={err:.5f} -> {status}")
    # 🧠 ML: the sensor with the largest error tells the engineer where to look
    sensors = ["vibration", "temperature", "pressure", "flow"]
    print(f"{'':<13} biggest error on: {sensors[np.argmax((x - rebuilt) ** 2)]}")

**Try this next:**
- Change the bearing-wear vibration from 0.90 to 0.60 - is it still caught?
- Make every sensor 0.95 at once (high but consistent load) and confirm it is NOT flagged.
- Change the threshold multiplier from 3 to 1.5 and see if the normal reading starts alarming.

## MEDIUM: PyTorch autoencoder flags valve leaks
Train a small PyTorch autoencoder on healthy compressor data and detect valve-leak readings by reconstruction error.

| 5W + H | |
|---|---|
| **Who** | A data scientist on the asset-integrity team. |
| **What** | 3,000 healthy readings of 6 sensors for training, plus new normal and faulty readings for testing. |
| **When** | When a compressor has enough healthy history but too few labelled failures for supervised learning. |
| **Where** | A Python job reading historian data, run on CPU. |
| **Why** | A leak can keep every sensor within its own limits while breaking the normal relationship between them. |
| **How** | Standardise with healthy statistics, train a 6-8-2-8-6 network to copy its input, set the threshold at the 99th percentile of healthy error. |

**🧠 ML concepts:** Training only on normal data, Bottleneck smaller than input, Mean squared error loss, Percentile-based threshold, False alarm rate vs detection rate  
**🐍 Python concepts:** Functions with default keyword arguments, pandas column-wise arithmetic, lambda functions, nn.Sequential layer stacking, torch.no_grad context manager

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

torch.set_num_threads(1)
torch.manual_seed(42)
rng = np.random.default_rng(42)

# --- 1. Simulate compressor sensor readings (one row per 10 minutes) ---
def make_readings(n, fault=False):
    # 🧠 ML: healthy sensors are driven by two hidden factors - load and ambient temperature
    load = rng.uniform(0.4, 1.0, n)
    ambient = rng.normal(80, 10, n)
    # 🐍 PY: a dict of column -> array; each sensor is a formula of the hidden drivers plus noise
    df = pd.DataFrame({
        "vibration_mm_s": 2.0 + 3.0 * load + rng.normal(0, 0.15, n),
        "discharge_temp_f": 150 + 60 * load + 0.5 * ambient + rng.normal(0, 2, n),
        "suction_pressure_psi": 400 - 50 * load + rng.normal(0, 4, n),
        "discharge_pressure_psi": 900 + 300 * load + rng.normal(0, 8, n),
        "flow_mmscfd": 20 + 40 * load + rng.normal(0, 1, n),
        "motor_current_a": 150 + 200 * load + rng.normal(0, 5, n),
    })
    if fault:
        # Fault: valve leak - flow drops and temperature climbs while load looks normal
        df["flow_mmscfd"] -= rng.uniform(4, 10, n)
        df["discharge_temp_f"] += rng.uniform(8, 20, n)
    return df

train_df = make_readings(3000)
normal_test = make_readings(500)
fault_test = make_readings(50, fault=True)
print(train_df.describe().loc[["mean", "std"]].round(1).to_string())

# --- 2. Scale using training (normal) data only ---
# 🐍 PY: pandas arithmetic is column-wise, so this standardises all 6 sensors at once
mu, sd = train_df.mean(), train_df.std()
# 🐍 PY: a lambda is a tiny one-line function
to_tensor = lambda d: torch.tensor(((d - mu) / sd).values, dtype=torch.float32)
X_train, X_norm, X_fault = to_tensor(train_df), to_tensor(normal_test), to_tensor(fault_test)

# --- 3. Autoencoder: 6 sensors -> 2-number bottleneck -> 6 sensors ---
# 🧠 ML: the bottleneck forces the net to learn only the 'normal' relationships between sensors
model = nn.Sequential(
    nn.Linear(6, 8), nn.Tanh(),
    nn.Linear(8, 2),            # encoder output = the code
    nn.Linear(2, 8), nn.Tanh(),
    nn.Linear(8, 6),            # decoder output = rebuilt sensors
)
opt = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

# 🧠 ML: input and target are the SAME - the model learns to copy through the bottleneck
for epoch in range(301):
    opt.zero_grad()
    loss = loss_fn(model(X_train), X_train)
    loss.backward()
    opt.step()
    if epoch % 100 == 0:
        print(f"epoch {epoch:3d}  train MSE {loss.item():.4f}")

# --- 4. Reconstruction error and threshold ---
def recon_error(X):
    # 🐍 PY: torch.no_grad() skips gradient bookkeeping during scoring
    with torch.no_grad():
        return ((model(X) - X) ** 2).mean(dim=1).numpy()

err_train = recon_error(X_train)
# 🧠 ML: alarm when error exceeds the 99th percentile of NORMAL training error
threshold = np.percentile(err_train, 99)
err_norm, err_fault = recon_error(X_norm), recon_error(X_fault)
print(f"\nThreshold (99th pct of normal error): {threshold:.4f}")
print(f"Median error  normal: {np.median(err_norm):.4f}   faulty: {np.median(err_fault):.4f}")

# --- 5. Results ---
false_alarms = (err_norm > threshold).mean()
detected = (err_fault > threshold).mean()
print(f"\nFalse alarm rate on new normal data: {false_alarms:.1%}")
print(f"Valve-leak readings detected:        {detected:.1%}")
print("\nMaintenance read-out: a high error means the sensors no longer agree with each other")
print("the way they do on a healthy compressor - send a technician before the valve fails.")

**Try this next:**
- Change the bottleneck from 2 to 6 neurons and see detection fall as the net learns to copy faults too.
- Use the 95th percentile as threshold and compare false alarms and detection.
- Make the leak milder (flow drop 2-4) and see how detection changes.

## HARD: Windowed autoencoder with alarm tuning and sensor blame
Detect injected compressor faults from 40-minute sensor windows, pick a threshold from validation percentiles, measure precision/recall and attribute alarms to sensors.

| 5W + H | |
|---|---|
| **Who** | The predictive-maintenance team, control-room operators and reliability engineers. |
| **What** | A time-aware autoencoder service with a chosen threshold, persistence rule, per-sensor explanations and saved artefacts. |
| **When** | Scoring every 10 minutes; retrained after overhauls or seasonal change. |
| **Where** | A streaming job beside the SCADA historian; alerts go to the alarm manager. |
| **Why** | Engineers need early, trustworthy alarms that say where to look, without flooding the control room. |
| **How** | Simulate autocorrelated healthy data, train on windows with validation-based early stopping, sweep percentile thresholds, add 3-of-5 persistence, score labelled fault episodes and compute error shares per sensor. |

**🧠 ML concepts:** Sliding windows for time-series context, Validation-based early stopping, Threshold selection by percentile sweep, Precision and recall on rare events, Alarm persistence to reduce false alarms, Per-feature reconstruction error attribution  
**🐍 Python concepts:** Generator expression unpacked into three variables, List of tuples driving a loop, pandas .loc slice assignment, np.stack and reshape, Class with two sub-networks, Rolling sums with pandas

In [ ]:
"""Compressor anomaly detection: windowed autoencoder, validation-percentile threshold,
per-sensor attribution, precision/recall on injected faults, alarm persistence."""
import os
import tempfile

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

torch.set_num_threads(1)
torch.manual_seed(42)
rng = np.random.default_rng(42)
SENSORS = ["vibration_mm_s", "discharge_temp_f", "suction_pressure_psi",
           "discharge_pressure_psi", "flow_mmscfd", "motor_current_a"]
WINDOW = 4  # readings per window (10-min readings -> 40 minutes)


def simulate(n_steps):
    """Time-ordered healthy data: load drifts smoothly (not random every row), plus sensor noise."""
    # 🧠 ML: realistic data is autocorrelated - neighbouring readings look alike
    load = np.empty(n_steps)
    load[0] = 0.7
    for t in range(1, n_steps):
        # mean-reverting load: pulled back toward 70% each step, plus a small random push
        load[t] = 0.7 + 0.97 * (load[t - 1] - 0.7) + rng.normal(0, 0.03)
    load = np.clip(load, 0.4, 1.0)
    ambient = 80 + 10 * np.sin(np.arange(n_steps) * 2 * np.pi / 144)  # daily cycle
    return pd.DataFrame({
        "vibration_mm_s": 2 + 3 * load + rng.normal(0, 0.15, n_steps),
        "discharge_temp_f": 150 + 60 * load + 0.5 * ambient + rng.normal(0, 2, n_steps),
        "suction_pressure_psi": 400 - 50 * load + rng.normal(0, 4, n_steps),
        "discharge_pressure_psi": 900 + 300 * load + rng.normal(0, 8, n_steps),
        "flow_mmscfd": 20 + 40 * load + rng.normal(0, 1, n_steps),
        "motor_current_a": 150 + 200 * load + rng.normal(0, 5, n_steps)})


def inject_faults(df):
    """Add three labelled fault episodes into a test period."""
    df, label = df.copy(), np.zeros(len(df), dtype=int)
    # 🐍 PY: a list of tuples (start, end, description, {sensor: offset}) drives the injection
    episodes = [(300, 340, "valve leak", {"flow_mmscfd": -7, "discharge_temp_f": 14}),
                (700, 760, "bearing wear", {"vibration_mm_s": 1.2}),
                (1100, 1130, "pressure sensor drift", {"discharge_pressure_psi": 60})]
    for start, end, _, offsets in episodes:
        for col, delta in offsets.items():
            # 🧠 ML: ramp the fault in gradually, like a real developing failure
            df.loc[start:end - 1, col] += np.linspace(0.3, 1.0, end - start) * delta
        label[start:end] = 1
    return df, label, episodes


def to_windows(arr, w=WINDOW):
    # 🐍 PY: stacking shifted slices builds rows of the last w readings, flattened side by side
    return np.stack([arr[i:len(arr) - w + 1 + i] for i in range(w)], axis=1).reshape(len(arr) - w + 1, -1)


class AE(nn.Module):
    def __init__(self, n_in, code=3):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(n_in, 12), nn.Tanh(), nn.Linear(12, code))
        self.dec = nn.Sequential(nn.Linear(code, 12), nn.Tanh(), nn.Linear(12, n_in))

    def forward(self, x):
        return self.dec(self.enc(x))


def train_ae(Xtr, Xval, epochs=150, batch=128):
    model = AE(Xtr.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=5e-3)
    best, state = np.inf, None
    for _ in range(epochs):
        for idx in torch.randperm(len(Xtr)).split(batch):
            opt.zero_grad()
            nn.functional.mse_loss(model(Xtr[idx]), Xtr[idx]).backward()
            opt.step()
        with torch.no_grad():
            v = nn.functional.mse_loss(model(Xval), Xval).item()
        # 🧠 ML: keep the weights with the best validation reconstruction (simple early stopping)
        if v < best:
            best, state = v, {k: t.clone() for k, t in model.state_dict().items()}
    model.load_state_dict(state)
    return model, best


def sensor_errors(model, X):
    """Squared error per window, averaged per SENSOR across the window's time steps."""
    with torch.no_grad():
        sq = ((model(X) - X) ** 2).numpy()
    # 🧠 ML: reshaping to (windows, time, sensors) lets us attribute the error to each sensor
    return sq.reshape(len(sq), WINDOW, len(SENSORS)).mean(axis=1)


def persist(flags, k=3, n=5):
    # 🧠 ML: alarm only if k of the last n windows are flagged - cuts one-off false alarms
    return pd.Series(flags).rolling(n, min_periods=1).sum().values >= k


def prf(pred, truth):
    tp = (pred & truth).sum()
    precision = tp / max(pred.sum(), 1)
    recall = tp / max(truth.sum(), 1)
    return precision, recall


if __name__ == "__main__":
    # 🧠 ML: train and validate on HEALTHY history only; faults only appear in the test period
    hist = simulate(5000)
    train, val = hist.iloc[:4000], hist.iloc[4000:]
    test, label, episodes = inject_faults(simulate(1400))
    mu, sd = train.mean(), train.std()
    scale = lambda d: ((d - mu) / sd).values
    Xtr, Xval, Xte = (torch.tensor(to_windows(scale(d)), dtype=torch.float32) for d in (train, val, test))
    # A window is faulty if its LAST reading is faulty (we alarm on the newest data)
    y_win = label[WINDOW - 1:].astype(bool)

    model, best_val = train_ae(Xtr, Xval)
    print(f"Windows train/val/test: {len(Xtr)}/{len(Xval)}/{len(Xte)}  best val MSE {best_val:.4f}")

    val_err = sensor_errors(model, Xval).mean(axis=1)
    test_sens = sensor_errors(model, Xte)
    test_err = test_sens.mean(axis=1)

    # 🧠 ML: threshold is a percentile of VALIDATION error - never tuned on the faults we test on
    print("\nPercentile  threshold  precision  recall  (raw / with 3-of-5 persistence)")
    for pct in [95, 99, 99.5]:
        thr = np.percentile(val_err, pct)
        raw = test_err > thr
        p1, r1 = prf(raw, y_win)
        p2, r2 = prf(persist(raw), y_win)
        print(f"{pct:>9}   {thr:8.4f}   {p1:.2f}/{p2:.2f}    {r1:.2f}/{r2:.2f}")

    thr = np.percentile(val_err, 99)
    alarms = persist(test_err > thr)
    print(f"\nChosen threshold: 99th percentile = {thr:.4f}")
    print("\nPer-episode detection and top sensor by error share:")
    for start, end, name, _ in episodes:
        # 🐍 PY: convert reading indices to window indices by subtracting WINDOW - 1
        s, e = start - (WINDOW - 1), end - (WINDOW - 1)
        hit = np.where(alarms[s:e])[0]
        delay = f"{hit[0] * 10} min" if len(hit) else "missed"
        share = test_sens[s:e].mean(axis=0) / test_sens[s:e].mean(axis=0).sum()
        top2 = np.argsort(share)[::-1][:2]
        blame = ", ".join(f"{SENSORS[i]} {share[i]:.0%}" for i in top2)
        print(f"  {name:<22} first alarm after {delay:<8} | {blame}")

    healthy_alarm_hours = (alarms & ~y_win).sum() * 10 / 60
    print(f"\nFalse-alarm time on healthy periods: {healthy_alarm_hours:.1f} h of {(~y_win).sum() * 10 / 60:.0f} h")

    path = os.path.join(tempfile.mkdtemp(), "compressor_ae.pt")
    torch.save({"model": model.state_dict(), "mu": mu.values, "sd": sd.values, "threshold": thr}, path)
    print(f"Saved model, scaler and threshold -> {os.path.basename(path)}")
    print("Safety note: alarms support, not replace, API 618/670 machinery protection trips;")
    print("retrain after overhauls and log every alarm outcome for threshold review.")

**Try this next:**
- Change WINDOW to 1 and then 8 and compare recall and detection delay.
- Shrink the code size from 3 to 1 and watch false alarms rise.
- Change persistence to 2-of-3 and find a better precision/recall balance for the control room.